# Spotify Streaming Audit

## Problema de negocio

Una plataforma de streaming musical cuenta con un catálogo de **1.994 canciones** sin criterios claros para decidir cuáles deberían formar parte de sus playlists premium.

El análisis preliminar muestra que una parte importante del catálogo tiene bajo rendimiento en términos de `popularity`, lo que puede afectar:

- la experiencia del usuario,
- la percepción de calidad del servicio premium,
- la retención,
- y el engagement.

---

## Objetivo del proyecto

El objetivo es identificar qué canciones, géneros y segmentos musicales deberían priorizarse para playlists premium usando análisis de datos y machine learning.

---

## Pregunta de negocio

**¿Qué canciones del catálogo deberían formar parte de playlists premium y bajo qué criterios debería tomarse esta decisión de forma sostenible?**

---

## Dataset utilizado

Se utilizará un único dataset:

**Spotify-2000 (1).csv**

El dataset contiene información sobre:

- canciones,
- artistas,
- géneros,
- año,
- atributos acústicos,
- y `popularity`.

# 1. Importación de librerías

Se importan las librerías necesarias para:

- limpieza de datos,
- análisis exploratorio,
- visualización,
- SQL,
- machine learning,
- clustering,
- y exportación de resultados.

In [1]:
import pandas as pd
import numpy as np
import re
import os

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    silhouette_score
)

import duckdb
import joblib

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

os.makedirs("visualizations", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


# 2. Carga del dataset

Sube manualmente el archivo:

`Spotify-2000 (1).csv`

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Spotify-2000 (1).csv to Spotify-2000 (1).csv


### Resultado esperado

Google Colab debe mostrar que el archivo `Spotify-2000 (1).csv` fue cargado correctamente.

In [3]:
df_raw = pd.read_csv("Spotify-2000 (1).csv")

print("Dimensiones del dataset original:")
print(df_raw.shape)

df_raw.head()

Dimensiones del dataset original:
(1994, 15)


,Index,Title,Artist,Top Genre,Year,Beats Per Minute (BPM),Energy,Danceability,Loudness (dB),Liveness,Valence,Length (Duration),Acousticness,Speechiness,Popularity
0,1,Sunrise,Norah Jones,adult standards,2004,157,30,53,-14,11,68,201,94,3,71
1,2,Black Night,Deep Purple,album rock,2000,135,79,50,-11,17,81,207,17,7,39
2,3,Clint Eastwood,Gorillaz,alternative hip hop,2001,168,69,66,-9,7,52,341,2,17,69
3,4,The Pretender,Foo Fighters,alternative metal,2007,173,96,43,-4,3,37,269,0,4,76
4,5,Waitin' On A Sunny Day,Bruce Springsteen,classic rock,2002,106,82,58,-5,10,87,256,1,3,59


# 3. Limpieza y estandarización del dataset

Para que el notebook sea reproducible y profesional, se realizará una limpieza inicial:

- nombres de columnas en formato `snake_case`,
- textos en minúsculas,
- eliminación de espacios innecesarios,
- conversión de duración a formato numérico,
- exportación del dataset limpio con separador de comas.

In [4]:
def to_snake_case(column_name):
    column_name = column_name.strip().lower()
    column_name = column_name.replace("(bpm)", "bpm")
    column_name = column_name.replace("(db)", "db")
    column_name = column_name.replace("(duration)", "duration")
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name


df = df_raw.copy()
df.columns = [to_snake_case(col) for col in df.columns]

df.columns

Index(['index', 'title', 'artist', 'top_genre', 'year', 'beats_per_minute_bpm',
       'energy', 'danceability', 'loudness_db', 'liveness', 'valence',
       'length_duration', 'acousticness', 'speechiness', 'popularity'],
      dtype='object')

### Resultado esperado

Los nombres de columnas quedan en formato limpio:

- `title`
- `artist`
- `top_genre`
- `year`
- `beats_per_minute_bpm`
- `energy`
- `danceability`
- `loudness_db`
- `liveness`
- `valence`
- `length_duration`
- `acousticness`
- `speechiness`
- `popularity`

In [5]:
text_columns = ['title', 'artist', 'top_genre']

for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

df[text_columns].head()

,title,artist,top_genre
0,sunrise,norah jones,adult standards
1,black night,deep purple,album rock
2,clint eastwood,gorillaz,alternative hip hop
3,the pretender,foo fighters,alternative metal
4,waitin' on a sunny day,bruce springsteen,classic rock


In [6]:
df['length_duration'] = (
    df['length_duration']
    .astype(str)
    .str.replace(',', '', regex=False)
    .astype(int)
)

df['length_duration'].head()

,length_duration
0,201
1,207
2,341
3,269
4,256


In [7]:
print("Valores nulos por columna:")
display(df.isnull().sum())

print("Duplicados:")
print(df.duplicated().sum())

Valores nulos por columna:


,0
index,0
title,0
artist,0
top_genre,0
year,0
beats_per_minute_bpm,0
energy,0
danceability,0
loudness_db,0
liveness,0


Duplicados:
0


In [8]:
df = df.drop_duplicates()

print("Dimensiones finales del dataset limpio:")
print(df.shape)

Dimensiones finales del dataset limpio:
(1994, 15)


In [9]:
os.makedirs("data_clean", exist_ok=True)

df.to_csv(
    "data_clean/spotify_2000_clean.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

print("Dataset limpio exportado correctamente en: data_clean/spotify_2000_clean.csv")

Dataset limpio exportado correctamente en: data_clean/spotify_2000_clean.csv


### Conclusión de limpieza

El dataset quedó estandarizado:

- columnas en `snake_case`,
- textos normalizados,
- duración como variable numérica,
- dataset limpio exportado con separador de comas.

A partir de este punto se trabajará con `df`.

# 4. Exploración inicial del dataset

Se revisa la estructura general del catálogo.

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   index                 1994 non-null   int64 
 1   title                 1994 non-null   object
 2   artist                1994 non-null   object
 3   top_genre             1994 non-null   object
 4   year                  1994 non-null   int64 
 5   beats_per_minute_bpm  1994 non-null   int64 
 6   energy                1994 non-null   int64 
 7   danceability          1994 non-null   int64 
 8   loudness_db           1994 non-null   int64 
 9   liveness              1994 non-null   int64 
 10  valence               1994 non-null   int64 
 11  length_duration       1994 non-null   int64 
 12  acousticness          1994 non-null   int64 
 13  speechiness           1994 non-null   int64 
 14  popularity            1994 non-null   int64 
dtypes: int64(12), object(3)
memory usage: 

In [11]:
df.describe()

,index,year,beats_per_minute_bpm,energy,danceability,loudness_db,liveness,valence,length_duration,acousticness,speechiness,popularity
count,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.00000
mean,997.500000,1992.992979,120.215647,59.679539,53.238215,-9.008526,19.012036,49.408726,262.443330,28.858074,4.994985,59.52658
std,575.762538,16.116048,28.028096,22.154322,15.351507,3.647876,16.727378,24.858212,93.604387,29.011986,4.401566,14.35160
min,1.000000,1956.000000,37.000000,3.000000,10.000000,-27.000000,2.000000,3.000000,93.000000,0.000000,2.000000,11.00000
25%,499.250000,1979.000000,99.000000,42.000000,43.000000,-11.000000,9.000000,29.000000,212.000000,3.000000,3.000000,49.25000
50%,997.500000,1993.000000,119.000000,61.000000,53.000000,-8.000000,12.000000,47.000000,245.000000,18.000000,4.000000,62.00000
75%,1495.750000,2007.000000,136.000000,78.000000,64.000000,-6.000000,23.000000,69.750000,289.000000,50.000000,5.000000,71.00000
max,1994.000000,2019.000000,206.000000,100.000000,96.000000,-2.000000,99.000000,99.000000,1412.000000,99.000000,55.000000,100.00000


### Hallazgo inicial

El dataset contiene variables útiles para evaluar canciones:

- `popularity`: métrica principal de rendimiento.
- `top_genre`: género musical.
- `year`: año de lanzamiento.
- `energy`, `danceability`, `loudness_db`, `acousticness`, `speechiness`: atributos acústicos.

La variable objetivo para el análisis y modelado será:

## `popularity`